# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Gustavo Crevelari (561408)
- Lucca Gomes (561996)
- Rafaela Ferreira (561671)
- Victor Sabelli (566224)

**Tema escolhido:** Assistente de saúde e bem-estar (IoB)

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [1]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [2]:
# from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

In [3]:
# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
# HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

GEMINI_API_KEY ok


In [4]:
# Modelos usados na atividade (os mesmos da Aula 05)
# MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite"

# cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [5]:
# >>> PERSONALIZE: descreva o assistente do grupo de acordo com o tema escolhido.
# Este é o "system": a instrução que define o comportamento do assistente.
# O exemplo abaixo é do tema 1 (casa inteligente). Troque pelo tema do grupo.

SYSTEM_PROMPT = """Você é um assistente de saúde e bem-estar conectado (IoB).
Ajude o usuário com dúvidas sobre monitoramento de sinais vitais, hábitos saudáveis e dispositivos de bem-estar.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com saúde e bem-estar, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Aqui o grupo vai ver, na prática, o efeito do `system`.
A **mesma pergunta** será enviada com **instruções de sistema diferentes**.

In [6]:
def perguntar_gemini(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,   # como o assistente deve se comportar
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [7]:
# >>> PERSONALIZE: crie 3 personalidades diferentes para o assistente do grupo.
personalidades = {
    "Especialista": SYSTEM_PROMPT,
    "Professor para crianças": "Você explica conceitos de saúde e bem-estar para crianças de 10 anos, usando comparações simples do dia a dia. Responda em português, em no máximo 4 frases.",
    "Resumido": "Você responde qualquer pergunta sobre saúde e bem-estar em uma única frase curta, em português.",
}

# >>> PERSONALIZE: uma pergunta relacionada ao tema do grupo.
pergunta = "Como os smartwatches conseguem monitorar a qualidade do sono e a frequência cardíaca?"

for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_gemini(pergunta, system))
    print()

===== Especialista =====
Os smartwatches monitoram a frequência cardíaca usando sensores de fotopletismografia, que brilham luzes LED na pele e medem as mudanças no fluxo sanguíneo. Já para o sono, eles combinam esses dados cardíacos com acelerômetros que detectam seus movimentos durante a noite. Com base nesses padrões, algoritmos avançados estimam o tempo que você passa em cada fase do sono, como o sono leve, profundo e REM. Essas métricas ajudam a entender melhor seus hábitos de descanso e a buscar uma rotina mais saudável. Lembre-se de que esses dispositivos oferecem estimativas úteis para o bem-estar, mas não substituem avaliações médicas profissionais.

===== Professor para crianças =====
Imagine que o smartwatch tem um super detetive e um pequeno tambor dentro dele. Para medir o coração, ele usa luzes que brilham na sua pele para sentir o sangue passando, como quando você coloca a lanterna do celular no dedo e vê ele ficar vermelhinho. E para o sono, ele funciona como um vigia q

**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade? Cada personalidade atendeu as suas instruções prévias, gerando um modelo de resposta diferente do anterior, por exemplo, o "Professor para crianças" utilizou de termos mais comuns e analogias simples para facilitar o entendimento do público-alvo.
- Qual `system` gerou a resposta mais útil para o tema? Por quê? O Especialista, como estamos trabalhando com informações voltadas a área da saúde é sempre necessário trazer uma resposta descritiva e analista para a pergunta, dessa forma garantimos melhor compreensão para o usuário e mais segurança no sistema.

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [8]:
def perguntar_gemini(pergunta, system):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,  # equivalente ao role "system"
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [9]:
# >>> PERSONALIZE: 3 perguntas sobre o tema do grupo.
perguntas = [
    "Quais métricas de saúde os smartwatches e anéis inteligentes costumam monitorar?",
    "Qual a diferença entre usar sensores de pulso e faixas peitorais para medir a frequência cardíaca?",
    "Quais cuidados de segurança e privacidade devo ter com dados de saúde armazenados em aplicativos?",
]

for p in perguntas:
    print("PERGUNTA:", p)
#     print("\n--- Hugging Face (Llama) ---")
#     print(perguntar_hf(p, SYSTEM_PROMPT))
    print("\n--- Gemini ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT))
    print("\n" + "=" * 60 + "\n")

PERGUNTA: Quais métricas de saúde os smartwatches e anéis inteligentes costumam monitorar?

--- Gemini ---
Os smartwatches e anéis inteligentes monitoram métricas essenciais como frequência cardíaca, variabilidade da frequência cardíaca (HRV) e qualidade do sono. Eles também costumam rastrear a saturação de oxigênio no sangue (SpO2), passos diários, calorias gastas e níveis de estresse. Alguns modelos mais avançados realizam eletrocardiogramas (ECG) e medem a temperatura corporal para prever ciclos menstruais. Esses dados ajudam você a entender melhor seus hábitos diários e a manter uma rotina mais saudável. Lembre-se de que essas medições são para bem-estar e não substituem exames médicos profissionais.


PERGUNTA: Qual a diferença entre usar sensores de pulso e faixas peitorais para medir a frequência cardíaca?

--- Gemini ---
Os sensores de pulso utilizam luz óptica (fotopletismografia) para estimar os batimentos na pele, sendo muito práticos para o uso diário e smartwatches. Já as 

**Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?  Testado somente com o Gemini, devido a erro da plataforma HF.

- Qual respondeu melhor? Em qual pergunta a diferença foi maior? As respostas do Gemini foram descritivas onde necessário, apesar de relativamente extensas abordam informações úteis em relação as perguntas. Não temos como comparar com o HF.

---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [11]:
# O Gemini usa os papéis "user" e "model"; o system vai em system_instruction
historico = []

config_chat = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    max_output_tokens=300,
)

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = []  # o system fica na config, então basta esvaziar a lista
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "parts": [{"text": entrada}]})

    try:
        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=historico,
            config=config_chat,
        )
        texto = resposta.text
    except Exception as e:
        historico.pop()  # remove a pergunta que falhou para não sujar o histórico
        print(f"\n(erro ao chamar o Gemini: {e})\n")
        continue

    historico.append({"role": "model", "parts": [{"text": texto}]})

    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico

Você: Eu me chamo Sabelli?

Assistente: Olá, Sabelli! Como seu assistente de saúde e bem-estar conectado (IoB), estou aqui para ajudar você a monitorar seus sinais vitais e adotar hábitos mais saudáveis. Como posso auxiliar no seu bem-estar hoje?

Você: Qual é meu nome?

Assistente: Você se chama Sabelli, conforme me disse anteriormente! Estou aqui para ajudar você a cuidar da sua saúde e monitorar seus hábitos diários. Como posso auxiliar no seu bem-estar hoje?

Você: limpar

(histórico apagado)

Você: Qual é meu nome?

Assistente: Como sou um assistente de saúde e bem-estar conectado (IoB), não tenho acesso a informações pessoais como o seu nome. Estou aqui para ajudar você a monitorar seus sinais vitais, sugerir hábitos saudáveis e tirar dúvidas sobre dispositivos de bem-estar. Como posso auxiliar na sua rotina de saúde hoje?

Você: sair
Encerrando chat.


In [12]:
# Veja como ficou a lista enviada ao modelo
print(f"[system] {SYSTEM_PROMPT[:80]}  (enviado em system_instruction)".replace(chr(10), " "))
for msg in historico:
    print(f"[{msg['role']}] {msg['parts'][0]['text'][:80]}".replace(chr(10), " "))

[system] Você é um assistente de saúde e bem-estar conectado (IoB). Ajude o usuário com d  (enviado em system_instruction)
[user] Qual é meu nome?
[model] Como sou um assistente de saúde e bem-estar conectado (IoB), não tenho acesso a 


**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`? Antes de limpar o assistente conseguiu retornar na conversa e buscar a referência do meu nome, respondendo que me chamo Sabelli. Depois de limpar ele perdeu essa referência e deu uma resposta padrão.
- Explique, com suas palavras, por que isso acontece. Isso acontece pois o modelo perde a referência marcada como "nome" dentro do histórico da conversa, pois ele foi apagado, ou seja não tem onde consultar.

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [13]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


# [HF DESATIVADO] versão original com seleção de provedor:
# def responder(mensagem, historico_gradio, provedor):
#     if provedor == "Hugging Face":
#         # Formato HF: roles system, user e assistant
#         mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
#         for msg in historico_gradio:
#             mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
#         mensagens.append({"role": "user", "content": mensagem})
#
#         resposta = cliente_hf.chat_completion(messages=mensagens, max_tokens=300)
#         return resposta.choices[0].message.content
#
#     else:
#         (ramo do Gemini, mantido abaixo)

def responder(mensagem, historico_gradio):
    # Formato Gemini: roles user e model; o system vai em system_instruction
    conteudos = []
    for msg in historico_gradio:
        papel = "model" if msg["role"] == "assistant" else "user"
        conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
    conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
    )
    return resposta.text

In [14]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    # [HF DESATIVADO] seletor de modelo:
    # additional_inputs=[gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo")],
    title="Assistente de Saúde e Bem-Estar (IoB)",
    description="Pergunte sobre monitoramento de sinais vitais, dispositivos conectados e hábitos saudáveis.",
    examples=[
        # [HF DESATIVADO] exemplos originais (com o provedor):
        # ["Como os smartwatches ajudam a monitorar o sono?", "Hugging Face"],
        # ["Quais cuidados ter com a privacidade de dados de saúde?", "Gemini"],
        ["Como os smartwatches ajudam a monitorar o sono?"],
        ["Quais cuidados ter com a privacidade de dados de saúde?"],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f29ff9a49599233ecb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê? Não foi possível alterar o modelo durante a conversa erro de pagamento no HF.

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [15]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from google import genai
from google.genai import types

# A chave e o system prompt vêm de variáveis de ambiente (nunca escreva a chave no código)
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELO = os.environ.get("MODELO_GEMINI", "gemini-3.5-flash-lite")
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.models.generate_content(
        model=MODELO,
        contents=pergunta.mensagem,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
        ),
    )
    return {"resposta": resposta.text}

Writing app.py


In [16]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["MODELO_GEMINI"] = MODELO_GEMINI
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)  # espera o servidor subir
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [17]:
# Testa o endpoint como um frontend faria (HTTP POST com JSON)
import requests

r = requests.post("http://localhost:8000/chat", json={"mensagem": "O que é IoT?"})
print(r.status_code)
print(r.json()["resposta"])

200
A Internet das Coisas (IoT) refere-se à rede de objetos físicos conectados à internet que coletam e trocam dados. No contexto da saúde, ela engloba dispositivos vestíveis que monitoram seus sinais vitais em tempo real. Como assistente de bem-estar, meu foco é ajudar você com esses dispositivos e com a promoção de hábitos saudáveis. Posso tirar alguma dúvida sobre o monitoramento da sua saúde hoje?


In [18]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.
